<a href="https://colab.research.google.com/github/cozen03/2026-Programming/blob/main/%ED%99%8D%ED%98%B8%EC%A4%80_%EA%B8%89%EC%8B%9D%EB%A9%94%EB%89%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:
import requests
import json
import re
from gtts import gTTS #TTS 불러옴
from IPython.display import Audio

# 1. 사용자로부터 조회할 특정 날짜 입력받기
search_date = input("조회할 급식 날짜를 YYYYMMDD 형태로 입력하세요 (예: 20260420): ")

# 2. 나이스(NEIS) 교육정보 개방포털 급식 식단 정보 API 엔드포인트
url = "https://open.neis.go.kr/hub/mealServiceDietInfo"

# 3. API 요청 파라미터 설정
params = {
    "KEY": "9a70b7ee08d04a95919104aca6db6d58",      #API 키를 입력하세요.
    "Type": "json",
    "pIndex": 1,
    "pSize": 100,
    "ATPT_OFCDC_SC_CODE": "T10",     # 시도교육청코드 (T10: 제주특별자치도교육청)
    "SD_SCHUL_CODE": "9290056",      # 행정표준학교코드 (대정고등학교)
    "MLSV_YMD": search_date,         # 💡입력받은 특정 날짜로 검색
    "MMEAL_SC_CODE": "3"             # 💡식사코드 2 = 중식, 식사코드 = 3 석식: 중식만 필터링해서 검색
}

# 4. API 호출
response = requests.get(url, params=params)

# 5. 결과 출력
if response.status_code == 200:
    data = response.json()

    # 데이터가 정상 반환되었는지 확인
    if "mealServiceDietInfo" in data:
        rows = data["mealServiceDietInfo"][1]["row"]

        # 특정 날짜의 중식 데이터이므로 결과가 1건으로 좁혀집니다.
        row = rows[0]
        #print(row)



        # 원본 식단 내용 태그 변경 및 알레르기 성분 번호 제거
        raw_dish = row["DDISH_NM"]
        dish = raw_dish.replace("<br/>", ", ")
        clean_dish = re.sub(r'\([\d\.]+\)', '', dish).replace("  ", " ").strip()

        list_dish = clean_dish.split(' , ')
        print(list_dish)
        tts_dish = f'오늘의 메뉴는'
        for i in list_dish:
          tts_dish += i
          tts_dish += ' '
        tts_dish += '입니다.'
        tts = gTTS(tts_dish, lang = 'ko')
        tts.save('output.mp3')
        display(Audio('output.mp3', autoplay = True))
    # API 자체에서 에러 메시지가 온 경우 (예: 키 오류)
    elif "RESULT" in data:
         print(f"\n⚠️ 데이터 요청 오류: {data['RESULT']['MESSAGE']}")

    # 해당하는 데이터가 아예 없는 경우 (주말, 휴일 등)
    else:
         print(f"\n입력하신 날짜({search_date})에는 중식 급식 데이터가 존재하지 않습니다.")
else:
    print(f"API 네트워크 호출 실패! (상태 코드: {response.status_code})")


조회할 급식 날짜를 YYYYMMDD 형태로 입력하세요 (예: 20260420): 20260506
['현미찹쌀밥', '들깨배추국s', '어묵볶이s', '돼지고기간장불고기s', '에그랑땡s', '배추김치', '마카다미아쿠키s']
